# Function Calling with Gemini

In [1]:
from google import genai
client = genai.Client()

In [2]:
import requests,json
def get_current_weather(city:str)->dict:
    """ can be used to get/fetch current weather information for a city name
    """
    api_key = "6a8b0ac166a37e2b7a38e64416b3c3fe"

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = response.content.decode()
    response = json.loads(response)
    output = {"City Name":city,"weather":response["weather"][0]['description'],
              "temperature":response['main']['temp'],
              "unit":"kelvin"}

    return output

In [3]:
get_current_weather("Bangalore")

{'City Name': 'Bangalore',
 'weather': 'clear sky',
 'temperature': 305.03,
 'unit': 'kelvin'}

In [5]:
get_current_weather("Mumbai")

{'City Name': 'Mumbai',
 'weather': 'smoke',
 'temperature': 306.14,
 'unit': 'kelvin'}

### Tool with LLM

- we need to provide the metadata about the tools/tool to the llm
- metadata needs to have 3 components
    - name of the function/tool - should be descriptive, clear
    - description of the function - clear with a sample example
    - arguments: args and its description with example
- OPENAPI format to define metadata

In [6]:
weather_tool = [{"name":"get_current_weather",
                 "description":"this function is used to get/fetch the current weather information for any given city.",
                 "parameters":{"type":"object",
                               "properties":{"city":{"type":'string','description':'name of any location/city. e.g. mumbai, new york'}},
                               "required":['city'],},
                               }]

In [7]:
from google.genai import types
import json
tools = types.Tool(function_declarations=weather_tool)
tool_map = {"get_current_weather":get_current_weather}

config = types.GenerateContentConfig(tools=[tools],
                                     automatic_function_calling=types.FunctionCallingConfig(mode="AUTO"))

Type mismatch in GenerateContentConfig.automatic_function_calling: expected AutomaticFunctionCallingConfig, got FunctionCallingConfig


In [14]:


def generate_content(prompt:str):
    content = [types.Content(role='user',parts=[types.Part(text=prompt)])]
    first_response = client.models.generate_content(model='gemini-2.5-flash',
                                                    contents=content, config = config)
    #print(first_response)
    if first_response.candidates[0].content.parts[0].function_call:
        # do something
        
        content.append(first_response.candidates[0].content)

        tool_calls = first_response.candidates[0].content.parts
        for tool in tool_calls:
            print("FC Decision: ",tool.function_call)
            tool_name = tool.function_call.name
            tool_args = tool.function_call.args
            fun_to_ex = tool_map[tool_name]
            tool_output = fun_to_ex(**tool_args)
            tool_response = types.Part.from_function_response(name=tool_name,response = {"result":tool_output})
            content.append(types.Content(role='user',parts=[tool_response]))
        
        second_response = client.models.generate_content(model='gemini-2.5-flash',contents=content)
        return second_response.text
        
        
    else:
        return first_response.text



In [15]:
generate_content("What is AI?")

'AI stands for Artificial Intelligence. It is a broad field of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence. This includes things like learning, problem-solving, decision-making, understanding language, and recognizing patterns.'

In [16]:
generate_content("Tell me weather in Delhi Today")

FC Decision:  id=None args={'city': 'Delhi'} name='get_current_weather' partial_args=None will_continue=None


'The weather in Delhi today is haze with a temperature of 307.2 Kelvin.'

In [17]:
generate_content("Tell me weather in Delhi and Mumbai Today")

FC Decision:  id=None args={'city': 'Delhi'} name='get_current_weather' partial_args=None will_continue=None
FC Decision:  id=None args={'city': 'Mumbai'} name='get_current_weather' partial_args=None will_continue=None


'The weather in Delhi is haze with a temperature of 307.2 Kelvin.\nThe weather in Mumbai is smoke with a temperature of 307.14 Kelvin.'

In [18]:
generate_content("explain quantum computing in 1 line")

'Quantum computing uses quantum-mechanical phenomena like superposition and entanglement to perform computations.'

In [19]:
results = generate_content("what is the weather in Dubai today")
results

FC Decision:  id=None args={'city': 'Dubai'} name='get_current_weather' partial_args=None will_continue=None


'The weather in Dubai today is scattered clouds with a temperature of 301.75 Kelvin.'

# Parallel and Multi Tool Calling

In [12]:
!pip install wikipedia --quiet


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [20]:
import wikipedia

def get_wikipedia_summary(query:str)->str:
    response = wikipedia.summary(query)
    return response

In [21]:
print(get_wikipedia_summary("Capital city of Egypt"))

The New Capital (Egyptian Arabic: العاصمة الجديدة, romanized: el-ʿĀṣima el-Gedīda) is a new urban community east of New Cairo in Cairo Governorate, Egypt. As of May 2023, 14 ministries and government entities have been relocated there. On 2 April 2024, president Abdel Fattah al-Sisi took the constitutional oath for a third consecutive term in office, officially inaugurating the new capital city as the seat of Egyptian government.


In [23]:
tool_def = [{"name":"get_current_weather",
                 "description":"this function is used to get/fetch the current weather information for any given city.",
                 "parameters":{"type":"object",
                               "properties":{"city":{"type":'string','description':'name of any location/city. e.g. mumbai, new york'}},
                               "required":['city'],},
                               },
            
            {"name":"get_wikipedia_summary",
                 "description":"this function is used to general and historical information about locations, people, places from wikipedia.",
                 "parameters":{"type":"object",
                               "properties":{"query":{"type":'string','description':'topic to search on wikipedia to get summary, e.g. capital of USA'}},
                               "required":['query'],},
                               }]

In [24]:
from google.genai import types
import json
tools = types.Tool(function_declarations=tool_def)
tool_map = {"get_current_weather":get_current_weather,
            "get_wikipedia_summary":get_wikipedia_summary}

config = types.GenerateContentConfig(tools=[tools],
                                     automatic_function_calling=types.FunctionCallingConfig(mode="AUTO"))

Type mismatch in GenerateContentConfig.automatic_function_calling: expected AutomaticFunctionCallingConfig, got FunctionCallingConfig


In [25]:
def generate_content(prompt:str):
    content = [types.Content(role='user',parts=[types.Part(text=prompt)])]

    while True:
        response = client.models.generate_content(model='gemini-2.0-flash',
                                                    contents=content,
                                                    config=config)
        if response.candidates[0].content.parts[0].function_call:
            print("LLM decided to make function call \n",response.candidates[0].content.parts)
            content.append(response.candidates[0].content)

            tool_calls = response.candidates[0].content.parts

            for tool_call in tool_calls:
                tool_name = tool_call.function_call.name
                tool_args = tool_call.function_call.args
                function_to_ex = tool_map[tool_name]
                tool_output = function_to_ex(**tool_args)
                tool_resp = types.Part.from_function_response(name=tool_name,response={"result":tool_output})
                content.append(types.Content(role='user',parts=[tool_resp]))
        else:
            break
    return response.text

In [26]:
generate_content("explain quantum computing in 1 line")

'Quantum computing uses quantum mechanics to solve complex problems faster than classical computers.\n'

In [28]:
generate_content("what is weather in delhi today")

LLM decided to make function call 
 [Part(
  function_call=FunctionCall(
    args={
      'city': 'delhi'
    },
    name='get_current_weather'
  )
)]


'OK. The current weather in Delhi is haze and the temperature is 307.2 Kelvin.'

In [30]:
# parallel function calls of the same function
generate_content("what is the weather in Singapore and Jakarta today.")

LLM decided to make function call 
 [Part(
  function_call=FunctionCall(
    args={
      'city': 'Singapore'
    },
    name='get_current_weather'
  )
), Part(
  function_call=FunctionCall(
    args={
      'city': 'Jakarta'
    },
    name='get_current_weather'
  )
)]


'The weather in Singapore is broken clouds and the temperature is 305.12 Kelvin. The weather in Jakarta is scattered clouds and the temperature is 306.89 Kelvin.'

In [31]:
# parallel function calls of different function
results = generate_content("what is the weather in Singapore and tell me more about singapore city.")
print(results)

LLM decided to make function call 
 [Part(
  function_call=FunctionCall(
    args={
      'city': 'Singapore'
    },
    name='get_current_weather'
  )
), Part(
  function_call=FunctionCall(
    args={
      'query': 'Singapore city'
    },
    name='get_wikipedia_summary'
  )
)]
The weather in Singapore is broken clouds and the temperature is 305.12 Kelvin.

Singapore, officially the Republic of Singapore, is an island country and city-state in Southeast Asia. Its territory comprises a main island, over 60 satellite islands and islets, and one outlying islet. The country is about one degree of latitude (137 kilometres or 85 miles) north of the equator, off the southern tip of the Malay Peninsula, bordering the Strait of Malacca to the west, the Singapore Strait to the south along with the Riau Islands in Indonesia, the South China Sea to the east and the Straits of Johor along with the State of Johor in Malaysia to the north.

In its early history, Singapore was a maritime emporium kn

In [37]:
# sequential function calls of different function
results = generate_content("what is the weather in the capital city of Tripura")
print(results)

LLM decided to make function call 
 [Part(
  function_call=FunctionCall(
    args={
      'query': 'capital of Tripura'
    },
    name='get_wikipedia_summary'
  )
)]
LLM decided to make function call 
 [Part(
  function_call=FunctionCall(
    args={
      'city': 'Agartala'
    },
    name='get_current_weather'
  )
)]
The current weather in Agartala, the capital city of Tripura, is haze with a temperature of 306.16 Kelvin.

